# Medically Constrained Counterfactual Explanations for Trustworthy Diabetes Risk Prediction

This notebook is a thin orchestration layer over the modules in `src/`. It calls the same, tested, leakage-safe code used by `scripts/run_full_pipeline.py` -- there is exactly one implementation of every step, not a notebook version and a separate script version.

**What changed vs. the original notebook** (see project README for the full list):
- Imputation (MICE) is fit only inside each CV fold, never on the full dataset before splitting.
- Hyperparameter tuning uses **nested CV** -- the search never sees the fold it is evaluated on.
- Every figure is saved to disk before `plt.show()` is called, in the same cell -- fixes the blank-PNG bug.
- Counterfactual recourse is generated by real optimization (`scipy.optimize.differential_evolution`), not hardcoded values, and propagates changes through a small causal DAG.
- Optuna's sampler is seeded -- reruns are reproducible.
- Calibration and fairness metrics use out-of-fold predictions, not in-sample.

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from src.config import DATA_PATH, RANDOM_STATE
from src import data_audit, nested_cv, stats_tests, calibration, shap_explain, fairness, plotting
from src import causal_recourse as cr

np.random.seed(RANDOM_STATE)

## Step 1 -- Data Audit

Quantify missingness, test the missingness mechanism, flag outliers, check duplicates and class balance, before any modeling decision is made.

In [ ]:
audit = data_audit.run_full_audit(DATA_PATH)
display(audit["zero_report"])
display(audit["missingness_mechanism"])
print("Duplicate rows:", audit["duplicates"])
display(audit["class_balance"])

In [ ]:
df = audit["raw"]
X = df.drop(columns=["Outcome"])
y = df["Outcome"]
plotting.plot_correlation_matrix(audit["clean_nan_encoded"])  # saved BEFORE show() internally

## Step 3 -- Baseline Model Comparison

Untuned baselines under leakage-safe repeated stratified CV (MICE fit inside each fold).

In [ ]:
baseline_results = nested_cv.run_baseline_cv(X, y)
baseline_summary = pd.DataFrame([
    {"Model": name, "Mean ROC-AUC": np.mean(r["auc"]), "Std ROC-AUC": np.std(r["auc"]),
     "Mean F1": np.mean(r["f1"]), "Mean Accuracy": np.mean(r["acc"])}
    for name, r in baseline_results.items()
]).sort_values("Mean ROC-AUC", ascending=False)
baseline_summary

## Step 4 -- Nested Cross-Validation (the tuning-leakage fix)

Hyperparameter search runs only on each outer training fold's own inner CV ; evaluation happens once, on the untouched outer test fold. This is the statistically valid replacement for the original notebook's single-loop `cross_val_score(pipeline, X, y, ...)` tuning, which let the eventual test set influence hyperparameter choice.

In [ ]:
nested_results = nested_cv.run_nested_cv_catboost(X, y)
print(f"Unbiased nested-CV mean ROC-AUC: {np.mean(nested_results['auc']):.4f} "
      f"+/- {np.std(nested_results['auc']):.4f}")

## Step 4b -- Paired Statistical Significance Test

In [ ]:
n = min(len(baseline_results["CatBoost"]["auc"]), len(nested_results["auc"]))
sig = stats_tests.paired_significance_test(baseline_results["CatBoost"]["auc"][:n], nested_results["auc"][:n])
sig

## Step 4c -- Final Deployment Model

One concrete model, tuned via CV on the full dataset, used only to build the explainability/calibration/recourse artifacts below. **Its own tuning score is not a performance claim** -- the unbiased estimate is the nested-CV result above.

In [ ]:
final_pipe, best_params = nested_cv.fit_final_deployment_model(X, y)
best_params

## Step 5 -- Calibration (out-of-fold, not in-sample)

In [ ]:
oof_proba = calibration.out_of_fold_probabilities(X, y, best_params)
bins, brier, ece = calibration.calibration_summary(y.values, oof_proba)
bin_df = pd.DataFrame(bins)
print(f"Brier score: {brier:.4f}   ECE: {ece:.4f}")
plotting.plot_calibration_curve(bin_df, brier, ece)

## Step 6 -- SHAP Explainability

Explains the SAME `final_pipe` used above for calibration -- not a separately-fit model instance.

In [ ]:
explainer, shap_values, X_display = shap_explain.compute_shap_values(final_pipe, X)
gi = shap_explain.global_importance_table(shap_values, X_display.columns.tolist())
gi

In [ ]:
plotting.plot_shap_summary(shap_values, X_display)
plotting.plot_shap_bar(shap_values, X_display)
plotting.plot_shap_dependence("Glucose", shap_values, X_display)

## Step 7 -- Causal-Aware Counterfactual Recourse

Real optimization (`differential_evolution`), not hardcoded values. Compares naive independent perturbation against a causally-propagated version.

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from src.preprocessing import ZeroToNaN

df_clean = ZeroToNaN().transform(X)
mice_completed = pd.DataFrame(
    IterativeImputer(max_iter=15, random_state=RANDOM_STATE).fit_transform(df_clean),
    columns=df_clean.columns,
)
equations = cr.fit_structural_equations(mice_completed)

proba_all = final_pipe.predict_proba(X)[:, 1]
high_risk_idx = int(np.argmax(proba_all))
patient = X.iloc[high_risk_idx].to_dict()
orig_proba = proba_all[high_risk_idx]

final_ind, proba_ind = cr.independent_recourse(final_pipe, patient, X.columns.tolist())
final_causal, proba_causal = cr.causal_recourse(final_pipe, patient, equations, X.columns.tolist())

print(f"Original risk: {orig_proba:.3f}")
print(f"Independent recourse risk: {proba_ind:.3f}")
print(f"Causal recourse risk: {proba_causal:.3f}")

In [ ]:
plotting.plot_recourse_comparison(orig_proba, proba_ind, proba_causal)

## Step 8 -- Fairness / Subgroup Analysis (out-of-fold probabilities)

In [ ]:
df_g = fairness.add_subgroups(df)
display(fairness.subgroup_performance(df_g, oof_proba, "Age_group"))
display(fairness.subgroup_performance(df_g, oof_proba, "Pregnancy_group"))